# 🚗 Matriculator · Pseudocódigo (Paso 3)

**Responsable:** Julen Altuna · **Lenguaje elegido para IA:** Python

Escenario: parking de una empresa con **empleados**, **clientes de un supermercado** (el cajero asocia la matrícula al **id del ticket**) y **público** que paga por tiempo. Este pseudocódigo describe **qué pasa cuando un coche llega a la salida**, que es donde se toman las decisiones (ver el diagrama 3.2 de la web).

> ⚠️ No se implementa ni se entrena ningún modelo. Para poder ejecutarlo, el detector, el OCR y las bases de datos están **simulados** con datos ficticios.

| Parte | Dónde está |
|---|---|
| 🟢 **Entrada** | `evento` de la cámara de salida (imagen + cámara + hora) y la confirmación humana |
| 🔵 **Funciones principales** | `leer_matricula()` (IA), `clasificar()` y `calcular_importe()` (reglas) |
| 🟠 **Condición / control de errores** | umbral de confianza, `if` de perfil y tiempo, `try/except` |
| 🩷 **Salida** | barrera / pago, aviso al personal y `resultado` guardado en CSV sin la imagen |

✏️ *TODO Julen: revisar y comentar con vuestras palabras.*

## 1 · Simulaciones (sustituyen al modelo real y a las bases de datos)

In [1]:
import csv, os, re
from datetime import datetime, date

class DetectorSimulado:                      # simula un detector tipo YOLO
    def detectar(self, imagen):
        return [{"caja": (212, 318, 398, 362), "confianza": imagen["conf"]}]

class OCRSimulado:                           # simula un OCR: "lee" el texto de la imagen ficticia
    def leer(self, imagen):
        return imagen["texto"]

detector, ocr = DetectorSimulado(), OCRSimulado()

# Bases de datos ficticias
BD_EMPLEADOS = {"1234BCD"}                                    # matrícula de empleados
BD_TICKETS = {("5678FGH", date(2026, 9, 23)): "T-2026-004817"} # (matrícula, día) -> id ticket
BD_ENTRADAS = {"1234BCD": datetime(2026, 9, 23, 8, 0),        # última hora de entrada
               "5678FGH": datetime(2026, 9, 23, 10, 0),
               "9012JKL": datetime(2026, 9, 23, 9, 30)}

def avisar_personal(evento, motivo="lectura dudosa"):
    print(f"   🔔 Aviso al personal ({evento['camara']}): {motivo}")

def esperar_confirmacion_humana(evento):
    return evento["imagen"]["texto_real"]   # la persona mira la cámara y confirma

## 2 · Pseudocódigo de la salida del parking

In [2]:
UMBRAL_CONFIANZA = 0.80
MIN_GRATIS_CLIENTE = 90       # minutos
TARIFA_HORA = 2.40            # €/hora
TOPE_DIARIO = 18.00           # €

def leer_matricula(imagen, detector, ocr):                # 🔵 parte de IA
    if imagen is None:
        raise ValueError("Imagen no válida")               # 🟠 error
    cajas = detector.detectar(imagen)
    if not cajas:
        return None, 0.0
    mejor = max(cajas, key=lambda c: c["confianza"])
    texto = re.sub(r"[^0-9A-Z]", "", ocr.leer(imagen).upper())
    return texto, mejor["confianza"]

def clasificar(matricula, dia):                           # 🔵 reglas
    if matricula in BD_EMPLEADOS:
        return "EMPLEADO", None
    id_ticket = BD_TICKETS.get((matricula, dia))
    if id_ticket:
        return "CLIENTE", id_ticket
    return "PUBLICO", None

def calcular_importe(perfil, minutos):                    # 🔵 reglas
    if perfil == "EMPLEADO":
        return 0.0
    if perfil == "CLIENTE":
        minutos = max(0, minutos - MIN_GRATIS_CLIENTE)
    return round(min(TOPE_DIARIO, minutos / 60 * TARIFA_HORA), 2)

def procesar_salida(evento):                              # 🟢 entrada: evento de la cámara
    try:
        matricula, conf = leer_matricula(evento["imagen"], detector, ocr)
        if matricula is None or conf < UMBRAL_CONFIANZA:  # 🟠 umbral
            avisar_personal(evento)                       # 🧑‍⚖️ revisión humana
            matricula = esperar_confirmacion_humana(evento)
        entrada = BD_ENTRADAS.get(matricula)
        if entrada is None:
            raise ValueError("Sin entrada registrada")
        perfil, id_ticket = clasificar(matricula, evento["hora"].date())
        minutos = int((evento["hora"] - entrada).total_seconds() // 60)
        importe = calcular_importe(perfil, minutos)
        resultado = {"matricula": matricula, "perfil": perfil, "id_ticket": id_ticket,
                     "minutos": minutos, "importe": importe, "estado": "OK"}
    except ValueError as e:                               # 🟠 control de errores
        avisar_personal(evento, str(e))
        resultado = {"estado": "ERROR", "mensaje": str(e)}
    resultado["hora"] = evento["hora"].isoformat(timespec="minutes")
    return resultado                                      # 🩷 salida

## 3 · Prueba con cuatro coches ficticios

In [3]:
salida = datetime(2026, 9, 23, 12, 30)
eventos = [
    {"camara": "CAM-SALIDA", "hora": salida, "imagen": {"texto": "1234 BCD", "conf": 0.95}},   # empleado
    {"camara": "CAM-SALIDA", "hora": salida, "imagen": {"texto": "5678 FGH", "conf": 0.91}},   # cliente, 150 min
    {"camara": "CAM-SALIDA", "hora": salida, "imagen": {"texto": "9O12 JKL", "conf": 0.55,     # lectura dudosa
                                                         "texto_real": "9012JKL"}},             #  -> la corrige una persona
    {"camara": "CAM-SALIDA", "hora": salida, "imagen": None},                                  # imagen no válida
]
registro = []
for ev in eventos:
    r = procesar_salida(ev)
    registro.append(r)
    print(r)

# 🩷 Registro mínimo en CSV (sin la imagen)
campos = ["hora", "matricula", "perfil", "id_ticket", "minutos", "importe", "estado", "mensaje"]
with open("movimientos.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=campos); w.writeheader(); w.writerows(registro)
print(open("movimientos.csv", encoding="utf-8").read())
os.remove("movimientos.csv")   # es solo una prueba

{'matricula': '1234BCD', 'perfil': 'EMPLEADO', 'id_ticket': None, 'minutos': 270, 'importe': 0.0, 'estado': 'OK', 'hora': '2026-09-23T12:30'}
{'matricula': '5678FGH', 'perfil': 'CLIENTE', 'id_ticket': 'T-2026-004817', 'minutos': 150, 'importe': 2.4, 'estado': 'OK', 'hora': '2026-09-23T12:30'}
   🔔 Aviso al personal (CAM-SALIDA): lectura dudosa
{'matricula': '9012JKL', 'perfil': 'PUBLICO', 'id_ticket': None, 'minutos': 180, 'importe': 7.2, 'estado': 'OK', 'hora': '2026-09-23T12:30'}
   🔔 Aviso al personal (CAM-SALIDA): Imagen no válida
{'estado': 'ERROR', 'mensaje': 'Imagen no válida', 'hora': '2026-09-23T12:30'}
hora,matricula,perfil,id_ticket,minutos,importe,estado,mensaje
2026-09-23T12:30,1234BCD,EMPLEADO,,270,0.0,OK,
2026-09-23T12:30,5678FGH,CLIENTE,T-2026-004817,150,2.4,OK,
2026-09-23T12:30,9012JKL,PUBLICO,,180,7.2,OK,
2026-09-23T12:30,,,,,,ERROR,Imagen no válida



## 4 · Explicación

- **`leer_matricula()`** es la única parte de **IA**: el detector localiza la matrícula y el OCR la lee. Devuelve el texto y la confianza.
- Si la confianza es **menor que 0,80**, el sistema no decide solo: **avisa al personal**, que confirma la matrícula (revisión humana).
- **`clasificar()`** y **`calcular_importe()`** son **reglas** sobre la base de datos: empleado → 0 €; cliente con ticket del día → gratis 90 min y después solo el exceso; público → 2,40 €/h con tope de 18 €.
- Los errores (imagen no válida, coche sin entrada registrada) se capturan con `try/except` y también avisan al personal.
- Solo se guarda un **registro mínimo** (hora, matrícula, perfil, ticket, minutos, importe), **nunca la imagen**.

✏️ *TODO Julen: ampliar con vuestras palabras.*